# Stage1 风险上限缺口：现金优势信息必要条件

假说、单分数、标签、预算和失败停止门见 plan.json；复用 Qlib Recorder，不重跑账户。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from scipy.stats import spearmanr
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
root = Path.cwd().resolve()
card = root / 'research/stage1_cap_gap_cash_information_20261023'
plan = json.loads((card / 'plan.json').read_text())
source = plan['sources']
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and plan['budget'] == {'new_market_requests':0,'model_fits':0,'native_accounts':0,'new_bootstraps':1,'new_qlib_recorders':1}
assert not (card / 'run_state.json').exists()
for prefix in ('raw','stage1','label'):
    source_card = root / source[prefix + '_card']
    assert sha(source_card / 'verification_manifest.json') == source[prefix + '_manifest_sha256']
    summary = json.loads((source_card / 'summary.json').read_text())
    assert summary['recorder_id'] == source[prefix + '_recorder_id']
    assert summary['nas_evidence']['sha256'] == source[prefix + '_NAS_archive_sha256']
    binding = json.loads((source_card / 'nas_binding.json').read_text())
    assert binding['evidence']['sha256'] == source[prefix + '_NAS_archive_sha256']
    manifest = json.loads((source_card / 'verification_manifest.json').read_text())
    artifact = {'raw':'inputs/frame','stage1':'paths/A_active_research_replay_0','label':'analysis/daily'}[prefix]
    expected = source[{'raw':'raw_frame_artifact_sha256','stage1':'stage1_path_artifact_sha256','label':'label_daily_artifact_sha256'}[prefix]]
    matches = [source_card / row['path'] for row in manifest['entries'] if row['path'].endswith('/' + artifact)]
    assert len(matches) == 1 and sha(matches[0]) == expected
if True:
    assert sha(root / source['stage1_card'] / 'readback_verification.json') == source['stage1_readback_sha256']
    assert sha(root / source['label_card'] / 'validation.json') == source['label_validation_sha256']
    assert json.loads((root / source['label_card'] / 'validation.json').read_text())['verified_objects'] == 12
qlib.init(provider_uri=str(root / 'data/qlib/stage1_domain_heads_20261018'), region='us')
assert len(D.calendar(start_time='2023-01-03', end_time='2026-10-09', freq='day')) >= 944

def load(prefix, artifact):
    sc = root / source[prefix + '_card']
    with R.uri_context(str(sc / 'qlib_recorder')):
        return R.get_recorder(experiment_name=sc.name, recorder_id=source[prefix + '_recorder_id']).load_object(artifact)

raw = load('raw','inputs/frame')
path = load('stage1','paths/A_active_research_replay_0')
labels = load('label','analysis/daily')
replay = path['replay']
assert len(raw) == len(replay) == len(labels) == 944
for other in (replay, labels):
    pd.testing.assert_frame_equal(raw[['signal_date','trade_date']].reset_index(drop=True),other[['signal_date','trade_date']].reset_index(drop=True))
daily = raw[['signal_date','trade_date','soxl_weight','soxx_weight']].merge(replay[['signal_date','trade_date','current_beta_cap','risk_state','execute_open_target']],on=['signal_date','trade_date'],validate='one_to_one').merge(labels[['signal_date','trade_date','cash_advantage_h5','label_available_h5','frozen_current_cash_weight']],on=['signal_date','trade_date'],validate='one_to_one')
daily['raw_beta'] = 3*daily.soxl_weight + daily.soxx_weight
daily['cap_gap'] = np.maximum(0, daily.raw_beta - daily.current_beta_cap)
daily['mature'] = daily.cash_advantage_h5.notna() & daily.label_available_h5.notna()
assert int(daily.mature.sum()) == 940 and daily.loc[~daily.mature].index.tolist() == [940,941,942,943]
assert np.isfinite(daily[['raw_beta','current_beta_cap','cap_gap']].to_numpy()).all()
assert (daily.loc[daily.mature,'label_available_h5'] > daily.loc[daily.mature,'trade_date']).all()

def rank_metric(group, score='cap_gap'):
    g = group.loc[group.mature]
    if len(g)<3 or g[score].nunique()<2 or g.cash_advantage_h5.nunique()<2:
        return {'calendar_rows':len(group),'mature_rows':len(g),'spearman':None,'status':'too_small_or_constant'}
    return {'calendar_rows':len(group),'mature_rows':len(g),'spearman':float(spearmanr(g[score],g.cash_advantage_h5).statistic),'status':'estimable'}

main = rank_metric(daily)
control = rank_metric(daily,'frozen_current_cash_weight')
annual = [{'year':str(y),'complete_year':str(y) in ('2024','2025'),**rank_metric(g)} for y,g in daily.groupby(daily.trade_date.dt.year,sort=True)]
quarter = [{'quarter':str(q),**rank_metric(g)} for q,g in daily.groupby(daily.trade_date.dt.to_period('Q'),sort=True)]
rolling = [{'window_index':i,'first_trade':str(g.trade_date.iloc[0].date()),'last_trade':str(g.trade_date.iloc[-1].date()),**rank_metric(g)} for i in range(len(daily)-125) for g in [daily.iloc[i:i+126]]]
assert len(annual)==4 and len(quarter)==16 and len(rolling)==819
assert main['status']=='estimable' and control['status']=='estimable'
bootstrap = plan['uncertainty']
state={'status':'bootstrap_attempted','plan_sha256':sha(card/'plan.json'),'new_model_fits':0,'new_native_accounts':0,'bootstrap_attempts':1}
atomic_json(state,card/'run_state.json')
values = daily.cap_gap.to_numpy(float)
target = daily.cash_advantage_h5.to_numpy(float)
n = len(daily)
rng = np.random.default_rng(bootstrap['seed'])
replicates = np.empty(bootstrap['draws'],dtype=float)
for k in range(bootstrap['draws']):
    idx = np.empty(n,dtype=np.int32)
    idx[0] = int(rng.integers(n))
    for j in range(1,n):
        idx[j] = int(rng.integers(n)) if rng.random()<1/bootstrap['mean_block'] else (idx[j-1]+1)%n
    mask = np.isfinite(target[idx])
    replicates[k] = spearmanr(values[idx][mask],target[idx][mask]).statistic
assert np.isfinite(replicates).all()
interval = [float(x) for x in np.quantile(replicates,bootstrap['two_sided_descriptive_interval'])]
necessary_support = bool(main['spearman']>0 and interval[0]>0)
result={'experiment':plan['experiment'],'decision':'descriptive_cap_gap_cash_rank_positive_needs_original39_incremental_test' if necessary_support else 'descriptive_cap_gap_cash_rank_necessary_condition_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'main':main,'matched_cash_control':control,'bootstrap_interval_95_descriptive':interval,'annual':annual,'quarter':quarter,'rolling_126_windows':len(rolling),'rolling_126_estimable':sum(x['status']=='estimable' for x in rolling),'cap_binding_days':int(daily.cap_gap.gt(1e-8).sum()),'necessary_support':necessary_support,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':1,'historical_independence':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/rolling','analysis/bootstrap_replicates']},'limitations':['Unadjusted descriptive interval after many prior research routes; no independent validation.','Own-state risk cap is endogenous to previous NAV and is not a causal treatment.','A positive rank does not establish incremental value beyond original Stage1 39 inputs or a fee-net SOXX account.']}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path='inputs',plan=plan,source_receipts={p:source[p+'_NAS_archive_sha256'] for p in ('raw','stage1','label')})
    recorder.save_objects(artifact_path='analysis',daily=daily,annual=annual,quarter=quarter,rolling=rolling,bootstrap_replicates=replicates,main=main,matched_cash_control=control,bootstrap_interval=interval)
    recorder.log_metrics(new_model_fits=0,new_native_accounts=0,cap_gap_spearman=main['spearman'],bootstrap_lower=interval[0])
    result.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id),recorder_uri=str(card/'qlib_recorder'))
atomic_json(result,card/'summary.json')
binding=json.loads((root/source['stage1_card']/'nas_binding.json').read_text())
binding.pop('evidence',None)
binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-09')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=result['recorder_id'],experiment_id=result['experiment_id'])
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
result['nas_evidence']=evidence
result['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(result,card/'summary.json')
state.update(status='completed',recorder_id=result['recorder_id'],experiment_id=result['experiment_id'])
atomic_json(state,card/'run_state.json')
q={row['quarter']:row for row in quarter}
(card/'conclusion.md').write_text(
    '# 原 Stage1 风险上限缺口对 SOXX 现金优势的排序\n\n'
    f'只读配对原 Stage1 0bp 完整路径、原始目标和已封存的五日现金优势标签；944个日历日全留，H5成熟940、末端4未知。风险上限压低原名义beta的日子 {result["cap_binding_days"]}/944。0拟合、0新账户、一次固定2000次/63日区块抽样。\n\n'
    f'主 Spearman 排序 {main["spearman"]:+.4f}，未经全项目研究选择校正的描述性95%区块区间 [{interval[0]:+.4f},{interval[1]:+.4f}]。同940日的原现金权重对照排序 {control["spearman"]:+.4f}。2024Q3与2026Q3分别为 {q["2024Q3"]["spearman"] if q["2024Q3"]["spearman"] is not None else float("nan"):+.4f} / {q["2026Q3"]["spearman"] if q["2026Q3"]["spearman"] is not None else float("nan"):+.4f}，仅作事后区间对照。所有4年度、16季度和819个连续126日窗口的样本数、未知及排序保存在Recorder。\n\n'
    f'决定 `{result["decision"]}`。上限由既往账户净值和风险状态内生决定；这不是取消上限的反事实收益，更不证明超出原39输入的信息增量。即使本必要条件为正，也只允许另冻匹配信息门，不能直接按缺口交易。2023–2026历史已复用；0bp全账户、2/5bp与独立前向仍须分别验证。Recorder `{result["recorder_id"]}`，NAS SHA `{remote}`。\n'
)
print(result['decision'],main['spearman'],interval,control['spearman'],result['recorder_id'])


descriptive_cap_gap_cash_rank_necessary_condition_failed -0.06221657820209111 [-0.22502791520632612, 0.07706379555242657] -0.018156445425217316 a13aeef00d184cbeaeef7b46f0f57db0


## 中文结论

原Stage1风险上限缺口对H5 SOXX现金优势的预定正向排序未成立：全期Spearman −0.0622，描述性95%区块区间[−0.2250,0.0771]。2024/2025两个完整年度点估计均负；2026部分年度为正不能用于换向或选窗。0拟合、0账户，13个Recorder工件经NAS限定恢复核验。此局部排序不是风控政策、完整费用账户或独立前向证据。